# Scalar X2 inference: H_agn total spline and H_para component model

## Aim

Fit the same total X2 WDM powers over $10^{-4}$--$10^{-1}\,\mathrm{Hz}$ with two models:

| Model | PSD model | Interpretation |
|---|---|---|
| H_agn | one log-P-spline for $S_{\rm total}$ | flexible recovery of the total local PSD; no component claim |
| H_para | free log-P-spline noise $+$ parametric Galaxy | component separation using a known Galactic response and weak scalar noise calibration |

Both likelihoods use only the total realization. Simulation truth is used to define the controlled-study pooling and response-null diagnostic and to evaluate recovery; it is never inserted as the fitted total or noise surface.

## Diagnostic convention

Every valid 0.1--100 mHz bin enters both fits. The visible few-mHz valley is a noise--Galaxy crossover and remains in every diagnostic. Genuine narrow high-frequency transfer minima also remain in the likelihood and surface-recovery errors; only their simulation-identified neighborhoods are omitted from broad mean-$z^2$ whitening summaries.


## 1. Statistical models

For real WDM coefficient $w_{nm}$ with local variance $S_{nm}$,

$$\log p(w\mid S)=-\frac12\sum_{n,m}\left[\log S_{nm}+\frac{w_{nm}^2}{S_{nm}}\right].$$

H_agn fits

$$S_{\rm total}^{(0)}(t,f)=\exp[B_t(t)W_0B_f(f)^\mathsf T].$$

H_para fits

$$\log S_{\rm noise}^{(1)}(t,f)=\log \mu_{\rm noise}+B_t(t)W_1B_f(f)^\mathsf T,$$

$$S_{\rm total}^{(1)}(t,f)=S_{\rm noise}^{(1)}(t,f)+A_{\rm gal}T_{\rm gal}(t,f;f_{\rm knee}).$$

$\mu_{\rm noise}$ is one scalar reference level, obtained here from the geometric median of an OMS/TM prediction over the analysis grid. It centres the proper spline prior but contributes no time- or frequency-dependent shape. The complete noise surface—including any transfer minima—must be learned from the total data. The Galactic response template is assumed known, while its amplitude and knee frequency are inferred jointly with the noise spline.

Both models use proper tensor-P-spline priors and two-chain NUTS. The additive variance decomposition is non-conjugate, so H_para is sampled jointly rather than by an exact Gibbs update.


In [ ]:
from pathlib import Path
import sys
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import binary_dilation, median_filter

ROOT = Path.cwd().resolve()
if not (ROOT / "combined_esa_xyz.h5").exists():
    ROOT = ROOT / "lisa_data_generation"
REPO = ROOT.parent / "wdm_psd"
if not (REPO / "tv_pspline_psd").exists():
    raise FileNotFoundError(f"Could not find tv_pspline_psd below {REPO}")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from tv_pspline_psd import (
    PSplineConfig,
    adaptive_frequency_bin_starts,
    fit_log_pspline_surface,
    summarize_mcmc_diagnostics,
    wdm_analysis_coefficients,
)
from tv_pspline_psd.adaptive_knots import fit_running_median_chi2_knots
from tv_pspline_psd.datasets import wdm_white_noise_calibration

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from component_pspline_fit import fit_component_pspline
from component_pspline_nuts import fit_component_pspline_nuts
from component_fit_diagnostics import masked_frequency_bin_mean, plot_component_model_comparison, plot_m1_parameter_posterior

ARCHIVE = ROOT / "combined_esa_xyz.h5"
print(f"archive: {ARCHIVE}")


## 2. Data and analysis policy

The archive supplies the total X2 time series. Both H_agn and H_para cover 0.1--100 mHz and use the same valid cells and adaptive frequency groups.

Three objects have deliberately different roles:

- `fit_mask`: every valid analysis bin; this enters the likelihood.
- `whitening_mask`: `fit_mask` minus a simulation-informed response-null neighborhood; this is used only for mean-$z^2$ summaries.
- deterministic component truth: used only for controlled-study pooling, diagnostic masking, and post-fit recovery metrics.

The null flag never edits the data, likelihood counts, fitted surface, or component-recovery metrics. It identifies narrow high-frequency transfer minima, not the broad few-mHz component-crossover valley.


In [ ]:
CHANNEL_INDEX = 0
CHANNEL_NAME = ("X2", "Y2", "Z2")[CHANNEL_INDEX]
START_DAY = 0.0
WINDOW_DAYS = None
NT = 32
FMIN_HZ, FMAX_HZ = 1.0e-4, 1.0e-1

# Settings selected by the preceding held-out investigation.
TIME_KNOTS, FREQ_KNOTS = 8, 60
N_WARMUP, N_SAMPLES, NUM_CHAINS = 250, 300, 2
CALIBRATION_DRAWS = 16

# Diagnostic-only response-null flag; these bins remain in the likelihood.
NULL_CONTINUUM_WINDOW_HZ = 5.0e-3
NULL_RELATIVE_THRESHOLD = 0.5
NULL_DILATION_FREQUENCY_BINS = 16

# Exact block-summed likelihood on a shared adaptive frequency grid.
# The posterior surface is still evaluated on all original WDM pixels.
LIKELIHOOD_MAX_LOG_RANGE = 0.08
LIKELIHOOD_MAX_FREQ_BIN = 64

with h5py.File(ARCHIVE, "r") as hdf:
    dt = float(hdf.attrs["dt_seconds"])
    t0_tcb = float(hdf.attrs["t0_tcb"])
    n_archive = int(hdf.attrs["n_samples"])
    truth_time_tcb = hdf["truth/time_tcb"][:]
    truth_frequency_hz = hdf["truth/frequency_hz"][:]

archive_days = n_archive * dt / 86400.0
print(f"{CHANNEL_NAME}: {n_archive:,} samples, dt={dt:g} s, span={archive_days:.3f} d")

In [ ]:
def wdm_valid_length(n_requested, nt):
    nf = n_requested // nt
    nf -= nf % 2
    if nt % 2 or nf < 2:
        raise ValueError("WDM requires even nt and an even N / nt >= 2")
    return nt * nf

start_index = int(round(START_DAY * 86400.0 / dt))
n_requested = n_archive if WINDOW_DAYS is None else int(round(WINDOW_DAYS * 86400.0 / dt))
n_total = wdm_valid_length(n_requested, NT)
stop_index = start_index + n_total
if start_index < 0 or stop_index > n_archive:
    raise ValueError("Requested window lies outside the archive")

with h5py.File(ARCHIVE, "r") as hdf:
    data = hdf["tdi/total"][CHANNEL_INDEX, start_index:stop_index]
assert data.shape == (n_total,) and np.isfinite(data).all()

nf = n_total // NT
df = 1.0 / (2.0 * nf * dt)
trim_low = max(1, int(np.ceil(FMIN_HZ / df)))
trim_high = max(0, nf - int(np.floor(FMAX_HZ / df)))
start_day_used = start_index * dt / 86400.0
end_day_used = stop_index * dt / 86400.0

FIT_TAG = f"{CHANNEL_NAME.lower()}_d{start_day_used:.0f}-{end_day_used:.0f}_fitall"
PLOTS_DIR = ROOT / "pspline_univar_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
SURFACE_PLOT_PATH = PLOTS_DIR / f"{FIT_TAG}_surface_comparison.png"
RESIDUAL_PLOT_PATH = PLOTS_DIR / f"{FIT_TAG}_residual_checks.png"
COMPONENT_PLOT_PATH = PLOTS_DIR / f"{FIT_TAG}_m0_m1_component_comparison.png"
H_PARA_PARAMETER_PLOT_PATH = PLOTS_DIR / f"{FIT_TAG}_m1_parameter_posterior.png"
COMPONENT_OUTPUT_PATH = ROOT / f"component_models_{FIT_TAG}_nuts.npz"
H_PARA_TIME_KNOTS, H_PARA_FREQUENCY_KNOTS = 5, 20
H_PARA_SMOOTHING_TIME, H_PARA_SMOOTHING_FREQUENCY = 30.0, 30.0
H_PARA_NOISE_LEVEL_LOG_SD = 5.0
H_PARA_N_WARMUP, H_PARA_N_SAMPLES, H_PARA_NUM_CHAINS = 500, 500, 2
H_PARA_TARGET_ACCEPT, H_PARA_MAX_TREE_DEPTH = 0.95, 12
print(f"days {start_day_used:.5g}--{end_day_used:.5g}; N={n_total:,}; nt={NT}; nf={nf:,}")
print(f"retained band: {trim_low * df:.3e}--{(nf - trim_high) * df:.3e} Hz")


## 3. Fit H_agn to all valid bins

The response-null flag is computed before inference but is not supplied to the likelihood. It is retained solely to define the later continuum-whitening diagnostic.

In [ ]:
config = PSplineConfig(
    n_interior_knots_time=TIME_KNOTS,
    n_interior_knots_freq=FREQ_KNOTS,
    trim_time_bins=1,
    trim_low_freq_channels=trim_low,
    trim_high_freq_channels=trim_high,
    freq_knot_strategy="linear",
    centered=True,
)

coefficients, fit_time_grid, fit_frequency_hz = wdm_analysis_coefficients(
    data, dt, NT, config
)


def truth_component_on_fit_grid(component):
    with h5py.File(ARCHIVE, "r") as hdf:
        source = hdf[f"truth/{component}_psd"][CHANNEL_INDEX]
    positive = source[source > 0.0]
    floor = positive.min() * 1.0e-6
    interpolator = RegularGridInterpolator(
        (truth_time_tcb, np.log(truth_frequency_hz)),
        np.log(np.maximum(source, floor)),
        bounds_error=False,
        fill_value=np.nan,
    )
    absolute_time_tcb = (
        t0_tcb
        + start_day_used * 86400.0
        + fit_time_grid * n_total * dt
    )
    time_mesh, frequency_mesh = np.meshgrid(
        absolute_time_tcb, np.log(fit_frequency_hz), indexing="ij"
    )
    points = np.column_stack((time_mesh.ravel(), frequency_mesh.ravel()))
    return np.exp(interpolator(points)).reshape(time_mesh.shape)


noise_truth = truth_component_on_fit_grid("noise")
truth_total = truth_component_on_fit_grid("total")

fit_df = float(np.median(np.diff(fit_frequency_hz)))
continuum_window_bins = max(
    5, int(round(NULL_CONTINUUM_WINDOW_HZ / fit_df))
)
continuum_window_bins += 1 - continuum_window_bins % 2
log_noise_truth = np.log(noise_truth)
# Process one WDM time row at a time to avoid a large 2-D median-filter temporary.
log_continuum = np.empty_like(log_noise_truth)
diagnostic_null_mask = np.empty_like(log_noise_truth, dtype=bool)
null_structure = np.ones(2 * NULL_DILATION_FREQUENCY_BINS + 1, dtype=bool)
for row in range(log_noise_truth.shape[0]):
    log_continuum[row] = median_filter(
        log_noise_truth[row], size=continuum_window_bins, mode="nearest"
    )
    null_core_row = (
        np.exp(log_noise_truth[row] - log_continuum[row])
        < NULL_RELATIVE_THRESHOLD
    )
    diagnostic_null_mask[row] = binary_dilation(null_core_row, structure=null_structure)
relative_response_proxy = np.exp(log_noise_truth - log_continuum)
fit_mask = np.ones_like(diagnostic_null_mask, dtype=bool)
whitening_mask = fit_mask & ~diagnostic_null_mask

print(
    f"whitening diagnostic omits {diagnostic_null_mask.sum():,}/"
    f"{diagnostic_null_mask.size:,} pixels ({diagnostic_null_mask.mean():.2%}); "
    "all remain in the fit"
)
assert fit_mask.shape == coefficients.shape
assert fit_mask.dtype == np.bool_
assert fit_mask.all() and diagnostic_null_mask.any() and whitening_mask.any()



selector_power = coefficients**2
knot_allocation = fit_running_median_chi2_knots(
    selector_power,
    fit_frequency_hz,
    FREQ_KNOTS,
    median_window_hz=5.0e-4,
)

# The archive truth defines this controlled-simulation pooling partition only.
pooling_pilot = np.log(truth_total)
frequency_bin_starts = adaptive_frequency_bin_starts(
    pooling_pilot, max_log_range=LIKELIHOOD_MAX_LOG_RANGE, max_bin=LIKELIHOOD_MAX_FREQ_BIN
)
print(f"likelihood frequency bins: {frequency_bin_starts.size:,} from {fit_frequency_hz.size:,} WDM channels")

t_fit = time.perf_counter()
fit = fit_log_pspline_surface(
    coefficients[None],
    fit_time_grid,
    fit_frequency_hz,
    config=config,
    interior_knots_freq=knot_allocation.knots,
    likelihood_mask=fit_mask,
    n_warmup=N_WARMUP,
    n_samples=N_SAMPLES,
    num_chains=NUM_CHAINS,
    random_seed=20260809,
    target_accept_prob=0.95,
    max_tree_depth=12,
    progress_bar=True,
    freq_bin_starts=frequency_bin_starts,
    binning_metadata={
        "selector": "archived_total_psd_oracle",
        "max_log_range": LIKELIHOOD_MAX_LOG_RANGE,
        "max_bin": LIKELIHOOD_MAX_FREQ_BIN,
    },
)
wall_seconds = time.perf_counter() - t_fit
diagnostics = summarize_mcmc_diagnostics(fit)
print(f"wall={wall_seconds:.1f} s; NUTS={fit['nuts_runtime_s']:.1f} s")
print(diagnostics)
print(fit["provenance"]["likelihood_mask"])

### Convert to one-sided PSD units

The posterior surface and inverse-PSD weights are retained across the full fitted band, including the dips. Recovery errors therefore expose any inability of the spline basis to follow the narrow response structure.

In [ ]:
calibration = wdm_white_noise_calibration(
    n_total, dt, NT, config, n_draws=CALIBRATION_DRAWS, seed=20260806
)
conversion = 2.0 * dt / calibration[None, :]
psd_spline_mean = np.asarray(fit["psd_mean"]) * conversion
psd_spline_lower = np.asarray(fit["psd_lower"]) * conversion
psd_spline_upper = np.asarray(fit["psd_upper"]) * conversion
time_days = start_day_used + np.asarray(fit["time_grid"]) * n_total * dt / 86400.0
frequency_hz = np.asarray(fit["freq_grid"])

psd_reported = np.where(fit_mask, psd_spline_mean, np.nan)
psd_reported_lower = np.where(fit_mask, psd_spline_lower, np.nan)
psd_reported_upper = np.where(fit_mask, psd_spline_upper, np.nan)
inverse_psd_weight = np.where(fit_mask, 1.0 / psd_spline_mean, 0.0)

comparison = fit_mask & np.isfinite(truth_total)
log_ratio = np.log(psd_spline_mean[comparison] / truth_total[comparison])
coverage90 = np.mean(
    (truth_total[comparison] >= psd_spline_lower[comparison])
    & (truth_total[comparison] <= psd_spline_upper[comparison])
)
print(f"fit-domain median |log(estimate / truth)| = {np.median(np.abs(log_ratio)):.3f}")
print(f"fit-domain 90% posterior-interval coverage = {coverage90:.3f}")
assert np.isfinite(psd_reported[diagnostic_null_mask]).all()
assert np.all(inverse_psd_weight[diagnostic_null_mask] > 0.0)

In [ ]:
figure_cmap = plt.get_cmap("magma").copy()
figure_cmap.set_bad("0.75")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), constrained_layout=True, sharey=True)
fields = (
    np.where(fit_mask, np.log10(psd_spline_mean), np.nan),
    np.where(fit_mask, np.log10(truth_total), np.nan),
    np.where(fit_mask, np.log10(psd_spline_mean / truth_total), np.nan),
)
for axis, field, title in zip(
    axes,
    fields,
    ("H_agn posterior mean", "generation truth", "log10(posterior / truth)"),
):
    image = axis.pcolormesh(
        time_days, frequency_hz, field.T, shading="auto", cmap=figure_cmap
    )
    axis.set(xlabel="time [days]", title=title, yscale="log")
    fig.colorbar(image, ax=axis, pad=0.01)
axes[0].set_ylabel("frequency [Hz]")
fig.suptitle(
    f"{CHANNEL_NAME} total PSD: every valid band pixel enters the likelihood", y=1.02
)
fig.savefig(SURFACE_PLOT_PATH, dpi=200, bbox_inches="tight")
print(f"saved surface comparison: {SURFACE_PLOT_PATH}")
plt.show()
plt.close(fig)

## 4. Fit H_para on the same full-band bins

H_para uses the identical fitted cells and frequency groups as H_agn. The archived OMS/TM reference is collapsed to one scalar prior centre before fitting; its surface is not passed to either optimizer or sampler. A penalized MAP estimate initializes two-chain NUTS, and component surfaces are reconstructed from the joint posterior draws.


In [ ]:
galactic_truth = truth_component_on_fit_grid("galactic")


def model_component_on_fit_grid(component):
    # Interpolate an archived model reference; never return a realization.
    with h5py.File(ARCHIVE, "r") as hdf:
        if "model" in hdf:
            source_time = hdf["model/time_tcb"][:]
            source_frequency = hdf["model/frequency_hz"][:]
            source = hdf[f"model/{component}"][CHANNEL_INDEX]
            reference_knee_hz = float(
                hdf["model"].attrs["galactic_template_reference_f_knee_hz"]
            )
            input_source = "archive model group"
        else:
            # Legacy controlled-archive fallback. The noise surface is reduced
            # to one scalar below and is never supplied to the likelihood.
            source_time = hdf["truth/time_tcb"][:]
            source_frequency = hdf["truth/frequency_hz"][:]
            if component == "noise_baseline_psd":
                source = hdf["truth/noise_psd"][CHANNEL_INDEX]
            elif component == "galactic_template_psd":
                source = (
                    hdf["truth/galactic_psd"][CHANNEL_INDEX]
                    / float(hdf.attrs["galactic_amplitude_scale"])
                )
            else:
                raise ValueError(component)
            reference_knee_hz = 2.15e-3
            input_source = "legacy controlled-archive reconstruction"
    positive = source[source > 0.0]
    interpolator = RegularGridInterpolator(
        (source_time, np.log(source_frequency)),
        np.log(np.maximum(source, positive.min() * 1.0e-6)),
        bounds_error=False,
        fill_value=np.nan,
    )
    absolute_time_tcb = t0_tcb + time_days * 86400.0
    time_mesh, frequency_mesh = np.meshgrid(
        absolute_time_tcb, np.log(frequency_hz), indexing="ij"
    )
    surface = np.exp(
        interpolator(
            np.column_stack((time_mesh.ravel(), frequency_mesh.ravel()))
        )
    ).reshape(time_mesh.shape)
    return surface, reference_knee_hz, input_source


noise_reference, reference_knee_hz, noise_reference_source = (
    model_component_on_fit_grid("noise_baseline_psd")
)
galactic_template, _, galactic_template_source = model_component_on_fit_grid(
    "galactic_template_psd"
)
observed_psd = coefficients**2 * conversion
component_fit_mask = fit_mask.copy()
noise_prior_level_psd = float(
    np.exp(np.median(np.log(noise_reference[component_fit_mask])))
)


def bin_for_m1(surface):
    return masked_frequency_bin_mean(
        surface, component_fit_mask, frequency_hz, frequency_bin_starts
    )


observed_binned, h_para_counts, h_para_frequency_hz = bin_for_m1(observed_psd)
galactic_template_binned, _, _ = bin_for_m1(galactic_template)
noise_truth_binned, _, _ = bin_for_m1(noise_truth)
galactic_truth_binned, _, _ = bin_for_m1(galactic_truth)
m0_total_binned, _, _ = bin_for_m1(psd_spline_mean)
m0_lower_binned, _, _ = bin_for_m1(psd_spline_lower)
m0_upper_binned, _, _ = bin_for_m1(psd_spline_upper)
null_fraction_binned, _, _ = bin_for_m1(diagnostic_null_mask.astype(float))

active_bins = np.sum(h_para_counts, axis=0) > 0.0
h_para_frequency_hz = h_para_frequency_hz[active_bins]
h_para_counts = h_para_counts[:, active_bins]
valid_m1 = h_para_counts > 0.0
h_para_whitening_mask = valid_m1 & (null_fraction_binned[:, active_bins] == 0.0)


def active_and_filled(surface, fallback):
    selected = surface[:, active_bins]
    return np.where(valid_m1, selected, fallback)


observed_binned = active_and_filled(observed_binned, 1.0)
galactic_template_binned = active_and_filled(galactic_template_binned, 0.0)
noise_truth_binned = active_and_filled(noise_truth_binned, np.nan)
galactic_truth_binned = active_and_filled(galactic_truth_binned, np.nan)
m0_total_binned = active_and_filled(m0_total_binned, np.nan)
m0_lower_binned = active_and_filled(m0_lower_binned, np.nan)
m0_upper_binned = active_and_filled(m0_upper_binned, np.nan)

h_para_map_fit = fit_component_pspline(
    observed_binned,
    t0_tcb + time_days * 86400.0,
    h_para_frequency_hz,
    noise_prior_level_psd=noise_prior_level_psd,
    galactic_template_psd=galactic_template_binned,
    reference_f_knee_hz=reference_knee_hz,
    effective_dof=np.where(valid_m1, h_para_counts, 1.0),
    mask=valid_m1,
    n_time_knots=H_PARA_TIME_KNOTS,
    n_frequency_knots=H_PARA_FREQUENCY_KNOTS,
    smoothing_time=H_PARA_SMOOTHING_TIME,
    smoothing_frequency=H_PARA_SMOOTHING_FREQUENCY,
    noise_level_log_sd=H_PARA_NOISE_LEVEL_LOG_SD,
)
h_para_posterior = fit_component_pspline_nuts(
    observed_binned,
    np.where(valid_m1, h_para_counts, 1.0),
    t0_tcb + time_days * 86400.0,
    h_para_frequency_hz,
    noise_prior_level_psd=noise_prior_level_psd,
    galactic_template_psd=galactic_template_binned,
    reference_f_knee_hz=reference_knee_hz,
    mask=valid_m1,
    n_time_knots=H_PARA_TIME_KNOTS,
    n_frequency_knots=H_PARA_FREQUENCY_KNOTS,
    noise_level_log_sd=H_PARA_NOISE_LEVEL_LOG_SD,
    n_warmup=H_PARA_N_WARMUP,
    n_samples=H_PARA_N_SAMPLES,
    num_chains=H_PARA_NUM_CHAINS,
    target_accept_probability=H_PARA_TARGET_ACCEPT,
    max_tree_depth=H_PARA_MAX_TREE_DEPTH,
    random_seed=20260810,
    progress_bar=True,
    map_fit=h_para_map_fit,
)
h_para_diagnostics = h_para_posterior.diagnostics
if (
    h_para_diagnostics["divergences"] > 0
    or not np.isfinite(h_para_diagnostics["max_r_hat"])
    or h_para_diagnostics["max_r_hat"] > 1.05
):
    raise RuntimeError(f"H_para posterior failed convergence gate: {h_para_diagnostics}")
if h_para_diagnostics["max_r_hat"] > 1.01:
    warnings.warn(
        f"H_para max R-hat exceeds 1.01: {h_para_diagnostics['max_r_hat']:.3f}"
    )
if h_para_diagnostics["min_ebfmi"] < 0.3:
    warnings.warn(
        f"H_para minimum E-BFMI is low: {h_para_diagnostics['min_ebfmi']:.3f}"
    )

h_para_noise_binned = np.where(valid_m1, h_para_posterior.noise_median, np.nan)
h_para_galactic_binned = np.where(valid_m1, h_para_posterior.galactic_median, np.nan)
h_para_amplitude = float(np.median(h_para_posterior.amplitude_draws))
h_para_f_knee_hz = float(np.median(h_para_posterior.f_knee_draws_hz))

comparison_metrics = plot_component_model_comparison(
    COMPONENT_PLOT_PATH,
    time_days=time_days,
    frequency_hz=h_para_frequency_hz,
    observed_total=observed_binned,
    counts=h_para_counts,
    truth_noise=noise_truth_binned,
    truth_galactic=galactic_truth_binned,
    m0_total=m0_total_binned,
    m0_lower=m0_lower_binned,
    m0_upper=m0_upper_binned,
    h_para_noise=h_para_noise_binned,
    h_para_galactic=h_para_galactic_binned,
    h_para_total_estimate=np.where(valid_m1, h_para_posterior.total_median, np.nan),
    galactic_amplitude=h_para_amplitude,
    f_knee_hz=h_para_f_knee_hz,
    h_para_noise_lower=np.where(valid_m1, h_para_posterior.noise_lower, np.nan),
    h_para_noise_upper=np.where(valid_m1, h_para_posterior.noise_upper, np.nan),
    h_para_galactic_lower=np.where(valid_m1, h_para_posterior.galactic_lower, np.nan),
    h_para_galactic_upper=np.where(valid_m1, h_para_posterior.galactic_upper, np.nan),
    h_para_total_lower=np.where(valid_m1, h_para_posterior.total_lower, np.nan),
    h_para_total_upper=np.where(valid_m1, h_para_posterior.total_upper, np.nan),
    h_para_diagnostics=h_para_diagnostics,
    whitening_mask=h_para_whitening_mask,
)
with h5py.File(ARCHIVE, "r") as hdf:
    injected_galactic_amplitude = float(hdf.attrs["galactic_amplitude_scale"])
plot_m1_parameter_posterior(
    H_PARA_PARAMETER_PLOT_PATH,
    amplitude_draws=h_para_posterior.amplitude_draws,
    f_knee_draws_hz=h_para_posterior.f_knee_draws_hz,
    phi_time_draws=np.exp(h_para_posterior.samples["phi_time"]),
    phi_frequency_draws=np.exp(h_para_posterior.samples["phi_freq"]),
    injected_amplitude=injected_galactic_amplitude,
    injected_f_knee_hz=reference_knee_hz,
)

h_para_input_source = (
    f"noise scalar from {noise_reference_source}; "
    f"Galactic template from {galactic_template_source}"
)
np.savez_compressed(
    COMPONENT_OUTPUT_PATH,
    time_days=time_days,
    frequency_hz=h_para_frequency_hz,
    requested_fmin_hz=FMIN_HZ,
    requested_fmax_hz=FMAX_HZ,
    fitted_frequency_min_hz=float(h_para_frequency_hz.min()),
    fitted_frequency_max_hz=float(h_para_frequency_hz.max()),
    counts=h_para_counts,
    whitening_mask=h_para_whitening_mask,
    observed_total=observed_binned,
    truth_noise=noise_truth_binned,
    truth_galactic=galactic_truth_binned,
    m0_total=m0_total_binned,
    m0_lower=m0_lower_binned,
    m0_upper=m0_upper_binned,
    h_para_noise=h_para_noise_binned,
    h_para_noise_lower=np.where(valid_m1, h_para_posterior.noise_lower, np.nan),
    h_para_noise_upper=np.where(valid_m1, h_para_posterior.noise_upper, np.nan),
    h_para_galactic=h_para_galactic_binned,
    h_para_galactic_lower=np.where(valid_m1, h_para_posterior.galactic_lower, np.nan),
    h_para_galactic_upper=np.where(valid_m1, h_para_posterior.galactic_upper, np.nan),
    h_para_total=np.where(valid_m1, h_para_posterior.total_median, np.nan),
    h_para_total_lower=np.where(valid_m1, h_para_posterior.total_lower, np.nan),
    h_para_total_upper=np.where(valid_m1, h_para_posterior.total_upper, np.nan),
    h_para_galactic_amplitude_draws=h_para_posterior.amplitude_draws,
    h_para_f_knee_hz_draws=h_para_posterior.f_knee_draws_hz,
    h_para_phi_time_draws=np.exp(h_para_posterior.samples["phi_time"]),
    h_para_phi_frequency_draws=np.exp(h_para_posterior.samples["phi_freq"]),
    noise_prior_level_psd=noise_prior_level_psd,
    noise_level_log_sd=H_PARA_NOISE_LEVEL_LOG_SD,
    model_definition=np.asarray(
        "free log-P-spline noise plus parametric Galactic foreground; "
        "no time-frequency OMS/TM noise template in likelihood"
    ),
    metric_names=np.asarray(list(comparison_metrics)),
    metric_values=np.asarray(list(comparison_metrics.values())),
    h_para_input_source=h_para_input_source,
    h_para_diagnostic_names=np.asarray(list(h_para_diagnostics)),
    h_para_diagnostic_values=np.asarray(list(h_para_diagnostics.values()), dtype=float),
)
print(
    {
        "H_para input source": h_para_input_source,
        "noise prior level": noise_prior_level_psd,
        "fitted band [Hz]": (h_para_frequency_hz.min(), h_para_frequency_hz.max()),
        "A_gal": h_para_amplitude,
        "f_knee_mHz": 1e3 * h_para_f_knee_hz,
        "H_para runtime_seconds": h_para_posterior.runtime_seconds,
        **h_para_diagnostics,
        **comparison_metrics,
    }
)
print(f"saved H_agn/H_para comparison: {COMPONENT_PLOT_PATH}")
print(f"saved H_para parameter posterior: {H_PARA_PARAMETER_PLOT_PATH}")
display(plt.imread(COMPONENT_PLOT_PATH))
display(plt.imread(H_PARA_PARAMETER_PLOT_PATH))


### Posterior figures and execution status

This cell writes the full-band H_agn/H_para component comparison, the shared Galactic-parameter posterior, and a versioned NPZ artifact. Do not reuse the earlier 20 mHz calibrated-baseline artifact: it represents a different model and band.

The production result is reportable only after the two-chain convergence gates in the preceding cell pass. A short run establishes execution plumbing but is not a posterior result for the manuscript.


## 5. Diagnostics

The fit-domain checks are surface error and posterior coverage. The whitening check is deliberately narrower only where a genuine response minimum is flagged:

$$z_{nm}^2=\frac{w_{nm}^2}{\widehat S_{nm}},$$

summarized over continuum cells away from flagged response-null neighborhoods. The few-mHz noise--Galaxy crossover remains included. This tests broad variance scale; it does not claim that null-core residuals are Gaussian or that the spline resolves a narrow transfer minimum.

Sampling is usable only with zero divergences, finite rank-normalized split $\hat R$, adequate ESS, no tree-depth saturation, and non-pathological E-BFMI.

In [ ]:
z2 = np.where(
    whitening_mask,
    np.asarray(fit["power"]) / np.asarray(fit["psd_mean"]),
    np.nan,
)
max_rhat = max(diagnostics["phi_time"]["r_hat"], diagnostics["phi_freq"]["r_hat"])
print(f"continuum mean z^2 = {np.nanmean(z2):.3f}")
print(f"divergences = {diagnostics['divergences']}; max R-hat(phi) = {max_rhat:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), constrained_layout=True)
axes[0].plot(time_days, np.nanmean(z2, axis=1), marker="o", ms=3)
axes[0].axhline(1.0, color="k", ls="--", lw=1)
axes[0].set(xlabel="time [days]", ylabel=r"continuum mean $z^2$", title="whitening by time (dip neighborhoods omitted)")
frequency_retained = whitening_mask.any(axis=0)
mean_z2_by_frequency = np.full(frequency_hz.shape, np.nan)
mean_z2_by_frequency[frequency_retained] = np.nanmean(
    z2[:, frequency_retained], axis=0
)
axes[1].semilogx(
    frequency_hz[frequency_retained],
    mean_z2_by_frequency[frequency_retained],
)
axes[1].axhline(1.0, color="k", ls="--", lw=1)
axes[1].set(xlabel="frequency [Hz]", ylabel=r"continuum mean $z^2$", title="whitening by frequency (dip neighborhoods omitted)")
fig.savefig(RESIDUAL_PLOT_PATH, dpi=200, bbox_inches="tight")
print(f"saved residual checks: {RESIDUAL_PLOT_PATH}")
plt.show()
plt.close(fig)

In [ ]:
output_path = ROOT / f"pspline_univar_{CHANNEL_NAME.lower()}_d{start_day_used:.0f}-{end_day_used:.0f}_fitall.npz"
np.savez_compressed(
    output_path,
    time_days=time_days,
    frequency_hz=frequency_hz,
    psd_spline_mean_hz2_per_hz=psd_spline_mean,
    psd_spline_lower_hz2_per_hz=psd_spline_lower,
    psd_spline_upper_hz2_per_hz=psd_spline_upper,
    psd_reported_hz2_per_hz=psd_reported,
    inverse_psd_weight=inverse_psd_weight,
    truth_total_hz2_per_hz=truth_total,
    fit_mask=fit_mask,
    whitening_mask=whitening_mask,
    diagnostic_null_mask=diagnostic_null_mask,
    relative_response_proxy=relative_response_proxy,
    z2_continuum=z2,
    null_continuum_window_hz=NULL_CONTINUUM_WINDOW_HZ,
    null_relative_threshold=NULL_RELATIVE_THRESHOLD,
    null_dilation_frequency_bins=NULL_DILATION_FREQUENCY_BINS,
    frequency_bin_starts=frequency_bin_starts,
    likelihood_grid_shape=np.asarray(fit["likelihood_grid_shape"]),
    diagnostics=np.array([diagnostics], dtype=object),
)
print(f"saved fit-all summary: {output_path}")
print(f"saved figures: {SURFACE_PLOT_PATH} and {RESIDUAL_PLOT_PATH}")

## 6. Interpretation and manuscript notes

### What each model means

- H_agn estimates only $S_{\rm total}(t,f)$. Agreement with the injected total PSD validates total-surface recovery, not a noise--foreground split.
- H_para estimates a free P-spline $S_{\rm noise}(t,f)$ and a parametric Galactic component with shared amplitude and knee parameters. The OMS/TM calculation supplies only one weak scalar level centre; its time-frequency shape is absent from the likelihood.
- The Galactic response and sky map remain conditional modelling assumptions.

### Diagnostics

All valid $10^{-4}$--$10^{-1}\,\mathrm{Hz}$ bins, including transfer minima and the few-mHz crossover, are fitted. Simulation-informed neighborhoods around genuine transfer minima are omitted only from broad whitening-scale summaries. Surface errors still include them. Posterior use requires zero divergences, finite rank-normalized split $\hat R$, adequate effective sample size, no material tree-depth saturation, and acceptable E-BFMI. Coverage from one realization is descriptive; calibration requires repeated simulations.

### What cannot yet be claimed

Total-PSD agreement alone does not identify the component split. Robustness to the weak noise-level prior, spline complexity and smoothing, Galactic response/sky mismatch, correlated A/E/T data, and random seed must be tested separately.

### Draft analysis paragraph

We transformed the total TDI time series into real WDM coefficients and modelled their local variances with a Whittle likelihood over $10^{-4}$--$10^{-1}\,\mathrm{Hz}$. Model H_agn represented the total PSD with a tensor-product log-P-spline. Model H_para decomposed the total variance into a free tensor-product log-P-spline instrumental-noise surface and an additive response-informed Galactic foreground with inferred amplitude and knee frequency. An analytic OMS/TM calculation was reduced to a single channel-level scale used to centre a weak prior; no analytic OMS/TM time-frequency surface entered the likelihood. All valid bins entered both likelihoods. Posterior inference used two-chain NUTS and was assessed using divergences, rank-normalized split $\hat R$, effective sample size, tree-depth saturation, and E-BFMI.
